## Models

**Goal**: Prepare dataset for ML modelling and start modelling

**Issues**:
* Encode the data_type and data_subject_type features
* Validate within the train set

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer

In [ ]:
# CSV file path
DF_HEALTH_PATH = "data-security-incident-trends-health-sector.csv"
# Target feature
TARGET_COL = "decision_taken"
# Multilabel columns contain a comma separated list of values and need multi label encoding
MULTILABEL_COLS = ["data_subject_type", "data_type"]
# Categorical columns are single value and need one hot encoding
CATEGORICAL_COLS = ["incident_category", "incident_type", "no_data_subjects_affected", "time_taken_to_report"]
# All feature columns
FEATURE_COLS = MULTILABEL_COLS + CATEGORICAL_COLS

# Time frame for test split - 2024 Q4, 2025 Q1 & Q2
TEST_TIMEFRAME = {
    "2024": ["Qtr 4"],
    "2025": ["Qtr 1", "Qtr 2"]
}

In [ ]:
# Load the preprocessed dataset
df = pd.read_csv(DF_HEALTH_PATH)

# Initial check for null values and data types - columns are int64 and object, no nulls
df.info()

In [ ]:
# define the train/ test period: train 2021 Q2 - 2024 Q1, test 2024 Q4 - 2025 Q1 & Q2
def test_period(row):
    return (((row["year"] == 2024) and (row["quarter"] in ["Qtr 4"])) or ((row["year"] == 2025) and (row["quarter"] in ["Qtr 1", "Qtr 2"])))

# temp column to check train vs test split
df["is_test"] = df.apply(test_period, axis=1)
print(df["is_test"].value_counts())

# split into train and test sets
df_train = df[df["is_test"] == False].reset_index(drop=True)
df_test = df[df["is_test"] == True].reset_index(drop=True)

# drop temp column but keep year and quarter for now to check distribution in train vs test sets
df = df.drop(columns=["is_test"])

# summary of years and quarters in train vs test sets
print("\nTraining set:\n", df_train[["year", "quarter"]].drop_duplicates().sort_values(["year", "quarter"]))
print("\nTest set:\n", df_test[["year", "quarter"]].drop_duplicates().sort_values(["year", "quarter"]))

# drop year, quarter and is_test columns for modeling
df_train = df_train.drop(columns=["year", "quarter", "is_test"])
df_test = df_test.drop(columns=["year", "quarter", "is_test"]) 

In [ ]:
# check distribution of target variable (decision taken) in original, train and test sets
print(df["decision_taken"].value_counts())
print(df_train["decision_taken"].value_counts())
print(df_test["decision_taken"].value_counts())

# proportion of decision taken in train vs test sets
print("\nTraining set decision_taken proportion:")
print(df_train["decision_taken"].value_counts(normalize=True))
print("\nTest set decision_taken proportion:")
print(df_test["decision_taken"].value_counts(normalize=True))

In [ ]:
# turn features into categorical for modeling (except data_subject_type and data_type)
for col in FEATURE_COLS + [TARGET_COL]:
    if col not in ["data_subject_type", "data_type"]:
        df_train[col] = df_train[col].astype("category")
        df_test[col] = df_test[col].astype("category")

# final check of new df structure
df.info()
df_train.info()
df_test.info()

In [ ]:
# function for evaluation metrics 
def evaluate_model(y_true, y_pred, labels=None):
    print("Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    
    # display confusion matrix
    print("Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(cm)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap=plt.cm.plasma)
    plt.title("Confusion Matrix")
    plt.show()
    
    print(f"Accuracy Score: {accuracy_score(y_true, y_pred)}")
    print(f"Precision: {precision_score(y_true, y_pred, average='macro', zero_division=0)}")
    print(f"Recall: {recall_score(y_true, y_pred, average='macro', zero_division=0)}")
    print(f"F1 Score: {f1_score(y_true, y_pred, average='macro', zero_division=0)}")

In [ ]:
# Training and test sets
X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET_COL]

X_test = df_test[FEATURE_COLS]
y_test = df_test[TARGET_COL]

print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])
print("Train:", y_train.value_counts().sort_index().to_dict())
print("Test:", y_test.value_counts().sort_index().to_dict())

In [ ]:
# BASELINE FOR RANDOM FOREST CLASSIFIER
preprocessor = ColumnTransformer(transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), FEATURE_COLS)])

rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced"))
    ]
)

rf_pipeline.fit(X_train, y_train)

y_pred = rf_pipeline.predict(X_test)
evaluate_model(y_test, y_pred)

In [ ]:
# extract feature importance to see what the model is learning from
rf_model = rf_pipeline.named_steps["model"]
encoded_model = rf_pipeline.named_steps["preprocess"].named_transformers_["cat"]
feature_names = encoded_model.get_feature_names_out(FEATURE_COLS)
feature_importances = rf_model.feature_importances_

# create a dataframe of feature importance and sort by importance
importances_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances
}).sort_values(by="importance", ascending=False)

# how many features are there after encoding
print(len(importances_df))

# display the top 20 most important features
importances_df.head(20)

# group the feature importance by original feature to see which features are most important overall
# separate incident type and category as they both start with incident, keep the original feature name for all other features
importances_df["original_feature"] = importances_df["feature"].apply(lambda x: x.split("_")[0] if not x.startswith("incident") else "_".join(x.split("_")[:2]))

feature_importance_summary = (
    importances_df
    .groupby("original_feature", observed=False)["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
print(feature_importance_summary)

# bar chart of feature importance summary
plt.figure(figsize=(10, 6))
bars = plt.bar(
    feature_importance_summary["original_feature"],
    feature_importance_summary["importance"],
    color= "#abc77d"
)
plt.title("Feature Importance Summary (Random Forest)", fontsize=14, fontweight='bold')
plt.xlabel("Original Feature", fontsize=12)
plt.ylabel("Total Importance", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True, axis='y', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()